In [1]:
import os
import pandas as pd
import sys

project_root = os.path.abspath("..")   # lên 1 cấp: MIND-research

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
project_root

'd:\\CDNC\\MIND-research'

In [3]:
from src.core.Context import VectorContext
from src.urv.URV import URV
from src.recommendation.RecommendationEngine import RecommendationEngine
from src.represent.RepresentedVector import RepresentedVector
from src.matrix.Matrix import Metrix
import numpy as np

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from src.matrix.Matrix import Metrix

In [4]:
# Semantic pretrained
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

# Topic model đã train
topic_model = BERTopic.load(project_root + "/models/primary/bertopic")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
impressions_path = os.path.join(
    project_root,
    "data",
    "primary",
    "test_set",
    "behaviors.tsv"
)

columns_impressions = [
    "behavior_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

impressions_df = pd.read_csv(
    impressions_path,
    sep="\t",
    names=columns_impressions,
)

impressions_df = impressions_df[
    ["behavior_id", "user_id", "history", "impressions"]
]

impressions_df = impressions_df.dropna(subset=["history"])
impressions_df = impressions_df.dropna(subset=["impressions"])

impressions_df = impressions_df.to_dict(orient="records")

In [6]:
context = VectorContext(os.path.join(project_root, "data", "primary", "test_set"))
display(len(impressions_df))

2341619

In [7]:
title_list = context.createTitleList()
title_list = {item["news_id"]: item["title"] for item in title_list}

represented_vector = RepresentedVector(semantic_model=semantic_model, topic_model=topic_model)
urv = URV()
recommender = RecommendationEngine(alpha=0.75)

In [9]:
behavior_news_ids = set()

for row in impressions_df:
    sample = context.createImpressionRow(row)

    behavior_news_ids.update(
        sample["history"].split()
    )
    behavior_news_ids.update(
        item["news_id"] for item in sample["impressions"]
    )

missing_titles = behavior_news_ids - set(title_list)

print("Tổng news_id trong history/impressions:", len(behavior_news_ids))
print("News_id không có title:", len(missing_titles))
print("Một số news_id thiếu:", list(missing_titles)[:10])

Tổng news_id trong history/impressions: 120944
News_id không có title: 2
Một số news_id thiếu: ['N1850', 'N89741']


In [14]:
impressions = {}
vector_cache = {}
missing_news = set()

def get_cached_vector(news_id):
    if news_id not in title_list:
        missing_news.add(news_id)
        return None

    if news_id not in vector_cache:
        vector_cache[news_id] = represented_vector.get_vector(
            title_list[news_id]
        )

    return vector_cache[news_id]

for row in impressions_df:
    sample = context.createImpressionRow(row)
    history_title_vector = {news_id: get_cached_vector(news_id)
                            for news_id in sample["history"].split(" ")
                            if news_id in title_list}
    
    sample_urv = urv.getURVFromVector(history_title_vector.values())
    cadidate_list = [{"news_id": item["news_id"],
                    "vector": get_cached_vector(item["news_id"]),
                    # "label": i["label"]
                    }
                    for item in sample["impressions"]
                    if item["news_id"] in title_list]
    impression = recommender.recommended_no_label(sample_urv, cadidate_list)
    impressions[row["behavior_id"]] = impression
    # results.append(Metrix(impression).evaluate())
    
# print("AUC:", np.mean([r["AUC"] for r in results]))
# print("MRR:", np.mean([r["MRR"] for r in results]))
# print("nDCG@5:", np.mean([r["nDCG@5"] for r in results]))
# print("nDCG@10:", np.mean([r["nDCG@10"] for r in results]))
    
    

KeyboardInterrupt: 

In [ ]:
print("Số behavior:", len(impressions))
print("Số vector đã cache:", len(vector_cache))

for row in impressions_df[:5]:
    expected = len(context.createImpressionRow(row)["impressions"])
    actual = len(impressions[row["behavior_id"]])
    print(row["behavior_id"], expected, actual)

In [ ]:
import pickle

output_path = os.path.join(
    project_root,
    "vectors",
    "primary",
    "matrix_test_results.pkl",
)

with open(output_path, "wb") as f:
    pickle.dump(impressions, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Đã lưu {len(impressions)} behavior vào:")
print(output_path)